# EDA Pipeline — Notebook de ejemplo

Este notebook ejecuta el pipeline de EDA a través de su API de Python (sin pasar por la CLI) sobre `data/raw/ecommerce.csv`, y muestra un resumen de los resultados.

Para el uso normal desde línea de comandos, vea `README.md` (comandos `analyze-file` / `analyze-batch`).

In [1]:
import json
from pathlib import Path

from eda_pipeline.config import Config
from eda_pipeline.pipeline import EDAPipeline

## Configurar y ejecutar el pipeline

`data/raw/ecommerce.csv` incluye una columna target binaria `is_churn`; el resto de las opciones (delimitador, codificación, separador decimal) se auto-detectan.

In [2]:
data_path = Path("..") / "data" / "raw" / "ecommerce.csv"

config = Config(
    input_file=str(data_path),
    output_dir=str(Path("..") / "reports"),
)
config.target.target_column = "is_churn"

pipeline = EDAPipeline(config)
pipeline.setup()
results = pipeline.run()

list(results.keys())

2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: EDA Pipeline started (correlation_id=d42f2a05)
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Loading data from: ..\data\raw\ecommerce.csv
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline.data_loader: Auto-detected format: csv
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline.data_loader: Loading CSV: ..\data\raw\ecommerce.csv (encoding=utf-8, delimiter=',', decimal='.')
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Analyzing ecommerce: 520 rows × 12 columns
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Inferring column types...
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline:   Numeric: 5, Categorical: 7, Text: 1, Datetime: 1
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Analyzing data quality...
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Performing univariate analysis...
2026-09-11 10:27:22 [d42f2a05] INFO     eda_pipeline: Analyzing relationships...
2026-09-11 10:27:22 [d42f2a

['ecommerce']

## Resumen de resultados

In [3]:
dataset_name, result = next(iter(results.items()))

print(f"Dataset: {dataset_name}")
print(f"Éxito: {result['success']}")
print(f"Alertas: {result['n_alerts']} | Gráficos: {result['n_plots']}")
print(f"Reporte HTML: {result['html_report']}")
print(f"Resumen JSON: {result['summary_json']}")
if result.get("failed_steps"):
    print(f"Pasos con error: {result['failed_steps']}")

Dataset: ecommerce
Éxito: True
Alertas: 5 | Gráficos: 30
Reporte HTML: ..\reports\ecommerce_20260911_102723\report.html
Resumen JSON: ..\reports\ecommerce_20260911_102723\summary.json


In [4]:
summary = json.loads(Path(result["summary_json"]).read_text(encoding="utf-8"))

print("Tipos de columnas inferidos:")
for col, dtype in summary["column_types"].items():
    print(f"  {col}: {dtype}")

print("\nAlertas principales:")
for alert in summary["alerts"][:5]:
    print(f"  [{alert['severity'].upper()}] {alert['message']}")

if "target" in summary:
    print("\nAnálisis de target:")
    print(f"  Columna: {summary['target']['target_column']} ({summary['target']['target_type']})")
    class_balance = summary["target"]["class_balance"]
    if class_balance:
        print(f"  Desbalanceado: {class_balance['is_imbalanced']} (conteos: {class_balance['counts']})")

Tipos de columnas inferidos:
  date: datetime
  customer_id: identifier
  age: numeric_continuous
  gender: categorical
  region: categorical
  product_category: categorical
  amount_spent: numeric_continuous
  items_purchased: numeric_discrete
  is_premium: boolean
  previous_purchases: numeric_continuous
  review_rating: numeric_discrete
  is_churn: boolean

Alertas principales:
  [HIGH] Desbalance de clases en target 'is_churn': la clase mayoritaria representa 83.2% de las muestras.
  [LOW] 20 filas duplicadas exactas (3.85%)
  [LOW] Columna 'amount_spent': 467 valores únicos (alta cardinalidad)
  [LOW] Columna 'customer_id': 500 valores únicos (alta cardinalidad)
  [LOW] Columna 'date': 500 valores únicos (alta cardinalidad)

Análisis de target:
  Columna: is_churn (classification)
  Desbalanceado: True (conteos: {'0.0': 417, '1.0': 84})


## Ver el reporte HTML

El reporte completo (auto-contenido, con todos los gráficos, tablas y alertas) está disponible en la ruta impresa arriba (`result["html_report"]`); ábralo directamente en el navegador.